In [5]:
import os
import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
TICK_SIZE   = np.float32(0.001)
BASE_DIR  = os.path.join(os.path.expanduser("~"), "Desktop", "my-first-project", "sample_data")
CLEAN_DIR = os.path.join(os.path.expanduser("~"), "Desktop", "my-first-project", "clean_data")
os.makedirs(CLEAN_DIR, exist_ok=True)

PRICE_COLS_L2 = ["last"] + [f"bp{i}" for i in range(1, 11)] + [f"ap{i}" for i in range(1, 11)]
VOL_COLS_L2   = [f"bv{i}" for i in range(1, 11)] + [f"av{i}" for i in range(1, 11)]

# ── Helper: 时间字符串 → 纯数字（8位，补前导零）────────────────────────────────
def parse_time_int(series: pd.Series) -> pd.Series:
    """
    兼容两种格式：
      '09:30:03.000'  → 93003000
      93003000        → 93003000（已是整数，直接补零对齐到8位）
    向量化字符串切片，避免逐行 apply。
    """
    s = series.astype(str).str.strip()

    # 判断是否含冒号（字符串格式）
    has_colon = s.str.contains(":", na=False)

    if has_colon.any():
        # '09:30:03.000' → 取 HH MM SS ms 各段拼接
        hh  = s.str[0:2]
        mm  = s.str[3:5]
        ss  = s.str[6:8]
        ms  = s.str[9:12].str.ljust(3, "0")   # 补足3位毫秒
        int_str = hh + mm + ss + ms
    else:
        # 纯数字字符串，补前导零到8位
        int_str = s.str.zfill(8)

    return int_str.astype("int32")


# ── Helper: 价格精度对齐（消除浮点误差）─────────────────────────────────────────
def align_price(series: pd.Series) -> pd.Series:
    """round to nearest tick, then cast to float32."""
    rounded = (series.astype("float64") / float(TICK_SIZE)).round() * float(TICK_SIZE)
    return rounded.astype("float32")


# ── 1. Level2 ─────────────────────────────────────────────────────────────────
def clean_level2() -> pd.DataFrame:
    path = os.path.join(BASE_DIR, "level2.csv")
    print(f"\n[level2] 读取 {path}")

    df = pd.read_csv(
        path,
        dtype={"date": "int32", "symbol": "category"},
    )

    # ① 时间标准化 + 前导零
    df["time"] = parse_time_int(df["time"])

    # ② 时间索引化
    df.set_index("time", inplace=True)
    df.sort_index(inplace=True)

    # ③ 单调性检查
    if not df.index.is_monotonic_increasing:
        print("  ⚠️  level2 时间轴不单调，已强制排序")

    # ④ 价格 float32（tick对齐）
    for col in PRICE_COLS_L2:
        df[col] = align_price(df[col])

    # ⑤ amount 保持 float64
    df["amount"] = df["amount"].astype("float64")

    # ⑥ 量 int32
    df["volume"] = df["volume"].astype("int32")
    for col in VOL_COLS_L2:
        df[col] = df[col].astype("int32")

    # ⑦ 去重（完全相同的行）
    before = len(df)
    df = df[~df.duplicated()]
    if len(df) < before:
        print(f"  去重删除 {before - len(df)} 行")

    # ⑧ 脏数据标记：bp1 >= ap1
    df["is_dirty"] = (df["bp1"] >= df["ap1"])
    dirty_count = df["is_dirty"].sum()
    if dirty_count:
        print(f"  ⚠️  发现 {dirty_count} 行脏数据（bp1 >= ap1），已标记 is_dirty=True")
    else:
        print("  ✓ 无价差穿越脏数据")

    # ⑨ 衍生字段：delta_volume（当前 volume − 上一快照 volume）
    df["delta_volume"] = df["volume"].diff().fillna(df["volume"]).astype("int32")

    out = os.path.join(CLEAN_DIR, "level2.parquet")
    df.to_parquet(out, engine="pyarrow", compression="snappy")
    print(f"  ✓ 输出 {out}  ({len(df):,} 行)")
    return df


# ── 2. Order ──────────────────────────────────────────────────────────────────
def clean_order() -> pd.DataFrame:
    path = os.path.join(BASE_DIR, "order.csv")
    print(f"\n[order] 读取 {path}")

    df = pd.read_csv(
        path,
        dtype={"date": "int32", "symbol": "category"},
    )

    # ① 时间标准化 + 前导零
    df["time"] = parse_time_int(df["time_str"])
    df.drop(columns=["time_str"], inplace=True)

    # ② 时间索引化
    df.set_index("time", inplace=True)
    df.sort_index(inplace=True)

    # ③ 单调性检查
    if not df.index.is_monotonic_increasing:
        print("  ⚠️  order 时间轴不单调，已强制排序")

    # ④ 价格 float32（tick对齐）
    df["price"] = align_price(df["price"])

    # ⑤ 量 int32
    df["quantity"] = df["quantity"].astype("int32")

    # ⑥ 分类变量
    df["side"]       = df["side"].astype("category")
    df["order_type"] = df["order_type"].astype("category")

    # ⑦ 去重
    before = len(df)
    df = df[~df.duplicated()]
    if len(df) < before:
        print(f"  去重删除 {before - len(df)} 行")

    out = os.path.join(CLEAN_DIR, "order.parquet")
    df.to_parquet(out, engine="pyarrow", compression="snappy")
    print(f"  ✓ 输出 {out}  ({len(df):,} 行)")
    return df


# ── 3. Trade ──────────────────────────────────────────────────────────────────
def clean_trade() -> pd.DataFrame:
    path = os.path.join(BASE_DIR, "trade.csv")
    print(f"\n[trade] 读取 {path}")

    df = pd.read_csv(
        path,
        dtype={"date": "int32", "symbol": "category"},
    )

    # ① 时间标准化 + 前导零
    df["time"] = parse_time_int(df["time_str"])
    df.drop(columns=["time_str"], inplace=True)

    # ② 时间索引化
    df.set_index("time", inplace=True)
    df.sort_index(inplace=True)

    # ③ 单调性检查
    if not df.index.is_monotonic_increasing:
        print("  ⚠️  trade 时间轴不单调，已强制排序")

    # ④ 价格 float32（tick对齐）
    df["price"] = align_price(df["price"])

    # ⑤ 量 int32
    df["quantity"] = df["quantity"].astype("int32")

    # ⑥ 去重（相同时间戳且内容完全一致视为重复成交记录）
    before = len(df)
    df = df[~df.duplicated()]
    if len(df) < before:
        print(f"  去重删除 {before - len(df)} 行")

    out = os.path.join(CLEAN_DIR, "trade.parquet")
    df.to_parquet(out, engine="pyarrow", compression="snappy")
    print(f"  ✓ 输出 {out}  ({len(df):,} 行)")
    return df


# ── 4. 跨表验证（merge_asof）─────────────────────────────────────────────────
def cross_validate(df_l2: pd.DataFrame, df_trade: pd.DataFrame):
    """
    用 merge_asof 将每笔 trade 对齐到最近的 level2 快照，
    检查成交价是否落在 [bp1, ap1] 区间内（宽松校验）。
    """
    print("\n[cross_validate] trade ↔ level2 价格区间校验")

    l2_ref = df_l2[["bp1", "ap1"]].reset_index()
    tr_ref = df_trade[["price"]].reset_index()

    merged = pd.merge_asof(
        tr_ref.sort_values("time"),
        l2_ref.sort_values("time"),
        on="time",
        direction="backward",
    )

    # 成交价应介于 bp1 和 ap1 之间（含边界）
    out_of_range = merged[
        (merged["price"] < merged["bp1"]) | (merged["price"] > merged["ap1"])
    ]
    if len(out_of_range):
        print(f"  ⚠️  {len(out_of_range)} 笔成交价超出盘口区间")
    else:
        print("  ✓ 所有成交价均在盘口区间内")


# ── Entry ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_l2    = clean_level2()
    df_order = clean_order()
    df_trade = clean_trade()
    cross_validate(df_l2, df_trade)
    print("\n✅ 全部清洗完成，Parquet 文件已写入:", CLEAN_DIR)


[level2] 读取 /Users/chongjidelaoshu/Desktop/my-first-project/sample_data/level2.csv


FileNotFoundError: [Errno 2] No such file or directory: '/Users/chongjidelaoshu/Desktop/my-first-project/sample_data/level2.csv'